# 01 – Explore QB dropbacks

Loads cached play-by-play data (2016–present) and filters it to QB dropbacks. Run `.venv/bin/python -m src.data.load` once first if `data/raw/` is empty.

In [1]:
import sys
from pathlib import Path

# Let the notebook import code from src/ (the notebook lives one folder down).
sys.path.insert(0, str(Path.cwd().parent))

import polars as pl
from src.data.load import load_pbp

pl.Config.set_tbl_cols(30)

polars.config.Config

In [2]:
pbp = load_pbp()
pbp.shape

(500466, 372)

## Filter to QB dropbacks

`qb_dropback == 1` covers pass attempts, sacks, and QB scrambles.

In [3]:
dropbacks = pbp.filter(pl.col("qb_dropback") == 1)
dropbacks.shape

(217923, 372)

## Columns most relevant to rating QBs

| column | meaning |
|---|---|
| `season`, `week`, `season_type` | when the play happened (`REG` / `POST`) |
| `game_id`, `posteam`, `defteam` | game, offense, defense |
| `passer_player_id`, `passer_player_name` | the QB on the dropback |
| `epa` | expected points added by the play (our target is next-week EPA per dropback) |
| `qb_epa` | EPA, but not penalizing the QB for a receiver's fumble after the catch |
| `cpoe` | completion % over expected (pass attempts only; null otherwise) |
| `pass_attempt`, `complete_pass`, `sack`, `qb_scramble`, `interception` | play outcome flags |
| `air_yards`, `yards_after_catch`, `yards_gained` | yardage |
| `success` | 1 if EPA > 0 |
| `down`, `ydstogo`, `wp` | situation: down, distance, win probability |

In [4]:
qb_cols = [
    "season", "week", "season_type", "game_id", "posteam", "defteam",
    "passer_player_id", "passer_player_name",
    "qb_dropback", "epa", "qb_epa", "cpoe",
    "pass_attempt", "complete_pass", "sack", "qb_scramble", "interception",
    "air_yards", "yards_after_catch", "yards_gained", "success",
    "down", "ydstogo", "wp",
]
qb = dropbacks.select(qb_cols)
qb.head(10)

season,week,season_type,game_id,posteam,defteam,passer_player_id,passer_player_name,qb_dropback,epa,qb_epa,cpoe,pass_attempt,complete_pass,sack,qb_scramble,interception,air_yards,yards_after_catch,yards_gained,success,down,ydstogo,wp
i32,i32,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2016,1,"""REG""","""2016_01_BUF_BAL""","""BAL""","""BUF""","""00-0026158""","""J.Flacco""",1.0,0.64454,0.64454,13.441056,1.0,1.0,0.0,0.0,0.0,-4.0,9.0,5.0,1.0,2.0,4.0,0.541975
2016,1,"""REG""","""2016_01_BUF_BAL""","""BAL""","""BUF""","""00-0026158""","""J.Flacco""",1.0,0.918295,0.918295,19.303071,1.0,1.0,0.0,0.0,0.0,1.0,8.0,9.0,1.0,2.0,10.0,0.52803
2016,1,"""REG""","""2016_01_BUF_BAL""","""BUF""","""BAL""","""00-0028118""","""T.Taylor""",1.0,-0.481404,-0.481404,20.712334,1.0,1.0,0.0,0.0,0.0,2.0,0.0,2.0,0.0,1.0,10.0,0.507005
2016,1,"""REG""","""2016_01_BUF_BAL""","""BUF""","""BAL""","""00-0028118""","""T.Taylor""",1.0,-1.374502,-1.374502,12.400633,1.0,1.0,0.0,0.0,0.0,-4.0,0.0,-4.0,0.0,3.0,14.0,0.447803
2016,1,"""REG""","""2016_01_BUF_BAL""","""BAL""","""BUF""","""00-0026158""","""J.Flacco""",1.0,-0.18526,-0.18526,23.208666,1.0,1.0,0.0,0.0,0.0,3.0,0.0,3.0,0.0,1.0,10.0,0.56639
2016,1,"""REG""","""2016_01_BUF_BAL""","""BAL""","""BUF""","""00-0026158""","""J.Flacco""",1.0,0.493441,0.493441,27.201819,1.0,1.0,0.0,0.0,0.0,5.0,1.0,6.0,1.0,2.0,7.0,0.559212
2016,1,"""REG""","""2016_01_BUF_BAL""","""BAL""","""BUF""","""00-0026158""","""J.Flacco""",1.0,2.476346,2.476346,38.063121,1.0,1.0,0.0,0.0,0.0,16.0,0.0,16.0,1.0,3.0,11.0,0.510546
2016,1,"""REG""","""2016_01_BUF_BAL""","""BAL""","""BUF""","""00-0026158""","""J.Flacco""",1.0,-0.456218,-0.456218,null,1.0,0.0,0.0,0.0,0.0,3.0,null,0.0,0.0,1.0,10.0,0.584447
2016,1,"""REG""","""2016_01_BUF_BAL""","""BUF""","""BAL""","""00-0028118""","""T.Taylor""",1.0,0.039256,0.039256,14.387792,1.0,1.0,0.0,0.0,0.0,-2.0,7.0,5.0,1.0,1.0,10.0,0.549582


In [5]:
qb.schema

Schema([('season', Int32),
        ('week', Int32),
        ('season_type', String),
        ('game_id', String),
        ('posteam', String),
        ('defteam', String),
        ('passer_player_id', String),
        ('passer_player_name', String),
        ('qb_dropback', Float64),
        ('epa', Float64),
        ('qb_epa', Float64),
        ('cpoe', Float64),
        ('pass_attempt', Float64),
        ('complete_pass', Float64),
        ('sack', Float64),
        ('qb_scramble', Float64),
        ('interception', Float64),
        ('air_yards', Float64),
        ('yards_after_catch', Float64),
        ('yards_gained', Float64),
        ('success', Float64),
        ('down', Float64),
        ('ydstogo', Float64),
        ('wp', Float64)])

# Phase 2: Explore + define the target

**Leakage guard:** every decision about the target is made by looking at the **training seasons only (2016–2023)**.
2024 (val) and 2025 (test) stay unseen, so their scores later are honest.

In [6]:
EXPLORE_SEASONS = list(range(2016, 2024))  # train only; do not widen this
train_pbp = pbp.filter(pl.col("season").is_in(EXPLORE_SEASONS))
train_pbp.select(pl.col("season").min().alias("first"), pl.col("season").max().alias("last"))

first,last
i32,i32
2016,2023


## Concept 1: one row per QB per game

Like turning a box of grocery receipts into one line per shopping trip: sort dropbacks into piles by
(game, team, passer), then write one summary line per pile with `dropbacks` and `epa_per_db`.
The code lives in `src/features/qb_games.py` so later phases reuse the exact same table.

In [7]:
from src.features.qb_games import qb_games

games = qb_games(train_pbp)

# How many team-games (one offense in one game) vs QB-games?
team_games = games.select(pl.struct("game_id", "posteam").n_unique()).item()
print(f"team-games: {team_games:,}   QB-games: {games.height:,}   ratio: {games.height / team_games:.2f}")

# How many different passers did each team-game have?
(games.group_by("game_id", "posteam").agg(pl.len().alias("passers"))
      .group_by("passers").agg(pl.len().alias("team_games")).sort("passers"))

team-games: 4,382   QB-games: 5,407   ratio: 1.23


passers,team_games
u32,u32
1,3431
2,880
3,68
4,3


### ▶ Play: Concept 1
Change a knob, rerun this cell, and compare against the default (`MIN_DROPBACKS = 1`).

In [10]:
# ---- KNOBS -----------------------------------------------------------------
MIN_DROPBACKS = 20                 # Try 3, then 10, then 20. Who disappears first? QBs or trick-play passers?
SHOW_GAME = "2023_01_DET_KC"      # Any game_id, format "<season>_<week 2 digits>_<AWAY>_<HOME>", e.g. "2022_10_SEA_TB"
# ---------------------------------------------------------------------------

kept = games.filter(pl.col("dropbacks") >= MIN_DROPBACKS)
dropped = games.filter(pl.col("dropbacks") < MIN_DROPBACKS)

print(f"Default (MIN_DROPBACKS=1): {games.height:,} rows")
print(f"Now (MIN_DROPBACKS={MIN_DROPBACKS}): {kept.height:,} rows  ->  dropped {dropped.height:,} "
      f"({dropped.height / games.height:.1%})")

if dropped.height:
    print("\nPassers who lost the most rows:")
    print(dropped.group_by("passer").agg(pl.len().alias("rows_dropped"), pl.col("dropbacks").sum().alias("dropbacks"))
                 .sort("rows_dropped", descending=True).head(10))

print(f"\nGame {SHOW_GAME} (rows marked kept=false would be dropped):")
games.filter(pl.col("game_id") == SHOW_GAME).with_columns(
    (pl.col("dropbacks") >= MIN_DROPBACKS).alias("kept")
).select("posteam", "passer", "dropbacks", "epa_sum", "epa_per_db", "kept")

Default (MIN_DROPBACKS=1): 5,407 rows
Now (MIN_DROPBACKS=20): 4,330 rows  ->  dropped 1,077 (19.9%)

Passers who lost the most rows:
shape: (10, 3)
┌───────────────┬──────────────┬───────────┐
│ passer        ┆ rows_dropped ┆ dropbacks │
│ ---           ┆ ---          ┆ ---       │
│ str           ┆ u32          ┆ u32       │
╞═══════════════╪══════════════╪═══════════╡
│ T.Hill        ┆ 54           ┆ 94        │
│ C.Rush        ┆ 16           ┆ 75        │
│ J.Brissett    ┆ 16           ┆ 65        │
│ J.Hekker      ┆ 16           ┆ 16        │
│ T.Taylor      ┆ 14           ┆ 122       │
│ P.Walker      ┆ 13           ┆ 142       │
│ B.Gabbert     ┆ 13           ┆ 90        │
│ B.Hoyer       ┆ 12           ┆ 71        │
│ J.Landry      ┆ 12           ┆ 12        │
│ R.Fitzpatrick ┆ 11           ┆ 141       │
└───────────────┴──────────────┴───────────┘

Game 2023_01_DET_KC (rows marked kept=false would be dropped):


posteam,passer,dropbacks,epa_sum,epa_per_db,kept
str,str,u32,f64,f64,bool
"""DET""","""J.Goff""",35,3.892088,0.111203,true
"""KC""","""P.Mahomes""",45,-3.528877,-0.078419,true


## Your exploration

Add cells below.